# FL Compression Study — Colab Sweep

**Results save to Google Drive automatically — safe to close the tab and come back.**

## How to use
1. Runtime → Change runtime type → **T4 GPU** → Save
2. Run **Cell 1** (mount Drive) — do this every time you reopen
3. Run **Cell 2** (install + setup) — do this every time you reopen
4. Run **Cell 3** anytime to check progress
5. Run whichever experiment cells you want — they auto-skip completed runs and auto-resume from checkpoints

## After disconnect / reopen
Just re-run Cells 1 and 2, then continue from whatever experiment cell you were on.
Checkpoints are saved to Drive so no work is lost.

## Results location on Drive
`My Drive / fl_compression_study / results /`  
You can view CSVs from any device at drive.google.com

In [ ]:
# ── Cell 1: Mount Drive ────────────────────────────────────────────────────────
# Run this every time you open the notebook.
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_BASE = '/content/drive/MyDrive/fl_compression_study'
os.makedirs(f'{DRIVE_BASE}/results', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/checkpoints/flower_cifar10', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/checkpoints/flower_cifar10_lr_decay', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/checkpoints/flower_cifar10_adaptive', exist_ok=True)

print('Drive mounted.')
print(f'Results will persist at: {DRIVE_BASE}/results/')

In [ ]:
# ── Cell 2: Install + Clone + Setup ───────────────────────────────────────────
# Run this every time you open the notebook.
import subprocess, sys, os

DRIVE_BASE = '/content/drive/MyDrive/fl_compression_study'
REPO_DIR   = '/content/fl-compression-study'
REPO       = 'https://github.com/ayananyways/fl-compression-study.git'

# System build tools (needed for pysz to compile)
subprocess.run(['apt-get', 'install', '-y', '-q', 'cmake', 'zlib1g-dev'], check=False)

# NumPy 2.0 removed np.float_ which flwr 1.9.0 dependencies use — pin to 1.x
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy<2.0'], check=True)

# flwr 1.9.0 = Python 3.12 compatible + has start_simulation
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'flwr[simulation]==1.9.0', 'pandas', 'seaborn'], check=True)

# pysz for SZ compressor runs
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pysz'], check=False)
SZ_OK = r.returncode == 0
print(f"pysz: {'OK' if SZ_OK else 'FAILED — SZ cells will not work'}")

# Clone or update repo
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-q', REPO, REPO_DIR], check=True)
    print('Repo cloned.')
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '-q'], check=False)
    print('Repo updated.')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# Symlink results + checkpoints → Drive so they persist across runtimes
def _link(local, drive_path):
    os.makedirs(drive_path, exist_ok=True)
    os.makedirs(os.path.dirname(local), exist_ok=True)
    if os.path.islink(local):
        os.unlink(local)
    elif os.path.isdir(local):
        import shutil
        for f in os.listdir(local):
            shutil.copy2(f'{local}/{f}', drive_path)
        shutil.rmtree(local)
    elif os.path.isfile(local):
        os.remove(local)
    os.symlink(drive_path, local)
    print(f'  linked: {local}')

_link(f'{REPO_DIR}/results',                              f'{DRIVE_BASE}/results')
_link(f'{REPO_DIR}/checkpoints/flower_cifar10',           f'{DRIVE_BASE}/checkpoints/flower_cifar10')
_link(f'{REPO_DIR}/checkpoints/flower_cifar10_lr_decay',  f'{DRIVE_BASE}/checkpoints/flower_cifar10_lr_decay')
_link(f'{REPO_DIR}/checkpoints/flower_cifar10_adaptive',  f'{DRIVE_BASE}/checkpoints/flower_cifar10_adaptive')

# Restart runtime so NumPy downgrade takes effect
print('\nInstall done. Restarting runtime to apply NumPy downgrade...')
import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)

In [ ]:
# ── Cell 3: Check progress (run anytime) ──────────────────────────────────────
import pandas as pd, os

files = [
    ('results/resnet20_cifar10_main.csv',     'Main sweep'),
    ('results/resnet20_cifar10_lr_decay.csv', 'LR decay'),
    ('results/resnet20_cifar10_adaptive.csv', 'Adaptive'),
]

for fpath, label in files:
    if os.path.exists(fpath):
        df = pd.read_csv(fpath)
        print(f'\n=== {label} ===')
        print(df.groupby(['compressor', 'seed'])['round'].agg(['count', 'max']))
    else:
        print(f'\n{label}: not started yet')

In [ ]:
# ── Cell 4: SZ main sweep ─────────────────────────────────────────────────────
# sz_eb0.001 x3 seeds + sz_eb0.01 x3 seeds
# Auto-skips runs already in CSV. Auto-resumes from checkpoint if partial.
import subprocess, sys

def run(cmd):
    print(f"\n>>> {' '.join(str(c) for c in cmd)}")
    r = subprocess.run(cmd, check=False)
    if r.returncode != 0:
        print(f'WARNING: exit {r.returncode} — checkpoint saved, continuing')

BASE = [
    sys.executable, 'fl-flower/run.py',
    '--dataset', 'cifar10',
    '--rounds', '200',
    '--num-clients', '10',
    '--alpha', '0.5',
    '--local-epochs', '2',
    '--output', 'results/resnet20_cifar10_main.csv',
]

print('=' * 55)
print('  SZ eb=0.001  x3 seeds')
print('=' * 55)
for seed in [0, 1, 2]:
    run(BASE + ['--compressor', 'sz', '--error-bound', '0.001', '--seed', str(seed)])

print('=' * 55)
print('  SZ eb=0.01  x3 seeds')
print('=' * 55)
for seed in [0, 1, 2]:
    run(BASE + ['--compressor', 'sz', '--error-bound', '0.01', '--seed', str(seed)])

print('\nSZ main sweep done.')

import pandas as pd
df = pd.read_csv('results/resnet20_cifar10_main.csv')
print(df.groupby(['compressor', 'seed'])['round'].agg(['count', 'max']))

In [ ]:
# ── Cell 5: LR decay sweep ────────────────────────────────────────────────────
# fp32+cosine, quant8+cosine, sz+cosine  x3 seeds each  = 9 runs
import subprocess, sys

def run(cmd):
    print(f"\n>>> {' '.join(str(c) for c in cmd)}")
    r = subprocess.run(cmd, check=False)
    if r.returncode != 0:
        print(f'WARNING: exit {r.returncode} — checkpoint saved, continuing')

BASE = [
    sys.executable, 'fl-flower/run.py',
    '--dataset', 'cifar10',
    '--rounds', '200',
    '--num-clients', '10',
    '--alpha', '0.5',
    '--local-epochs', '2',
    '--output', 'results/resnet20_cifar10_lr_decay.csv',
    '--lr-decay',
]

print('=' * 55)
print('  fp32 + cosine LR  x3 seeds')
print('=' * 55)
for seed in [0, 1, 2]:
    run(BASE + ['--compressor', 'none', '--seed', str(seed)])

print('=' * 55)
print('  quant_8bit + cosine LR  x3 seeds')
print('=' * 55)
for seed in [0, 1, 2]:
    run(BASE + ['--compressor', 'quantization', '--bits', '8', '--seed', str(seed)])

print('=' * 55)
print('  SZ eb=0.001 + cosine LR  x3 seeds')
print('=' * 55)
for seed in [0, 1, 2]:
    run(BASE + ['--compressor', 'sz', '--error-bound', '0.001', '--seed', str(seed)])

print('\nLR decay sweep done.')

import pandas as pd
df = pd.read_csv('results/resnet20_cifar10_lr_decay.csv')
print(df.groupby(['compressor', 'seed'])['round'].agg(['count', 'max']))

In [ ]:
# ── Cell 6: Adaptive SZ sweep ─────────────────────────────────────────────────
# Schedule A, B, C, A+cosine  x3 seeds each  = 12 runs
# Schedule C is the novel contribution (plateau-triggered error bound)
import subprocess, sys

def run(cmd):
    print(f"\n>>> {' '.join(str(c) for c in cmd)}")
    r = subprocess.run(cmd, check=False)
    if r.returncode != 0:
        print(f'WARNING: exit {r.returncode} — checkpoint saved, continuing')

BASE = [
    sys.executable, 'fl-flower/run.py',
    '--dataset', 'cifar10',
    '--rounds', '200',
    '--num-clients', '10',
    '--alpha', '0.5',
    '--local-epochs', '2',
    '--output', 'results/resnet20_cifar10_adaptive.csv',
]

for schedule in ['A', 'B', 'C']:
    print('=' * 55)
    print(f'  Schedule {schedule}  x3 seeds')
    print('=' * 55)
    for seed in [0, 1, 2]:
        run(BASE + ['--schedule', schedule, '--seed', str(seed)])

print('=' * 55)
print('  Schedule A + cosine LR  x3 seeds  (control)')
print('=' * 55)
for seed in [0, 1, 2]:
    run(BASE + ['--schedule', 'A', '--lr-decay', '--seed', str(seed)])

print('\nAdaptive sweep done.')

import pandas as pd
df = pd.read_csv('results/resnet20_cifar10_adaptive.csv')
print(df.groupby(['compressor', 'seed'])['round'].agg(['count', 'max']))

In [ ]:
# ── Cell 7: Final summary ─────────────────────────────────────────────────────
import pandas as pd, os

DRIVE_BASE = '/content/drive/MyDrive/fl_compression_study'
all_dfs = []

files = [
    ('results/resnet20_cifar10_main.csv',     'Main sweep'),
    ('results/resnet20_cifar10_lr_decay.csv', 'LR decay'),
    ('results/resnet20_cifar10_adaptive.csv', 'Adaptive'),
]

for fpath, label in files:
    if os.path.exists(fpath):
        df = pd.read_csv(fpath)
        all_dfs.append(df)
        done = df.groupby(['compressor', 'seed'])['round'].max()
        n_complete = (done >= 200).sum()
        print(f'\n=== {label} — {n_complete}/{len(done)} runs complete ===')
        print(df.groupby(['compressor', 'seed'])['round'].agg(['count', 'max']))
    else:
        print(f'\n{label}: not started')

print(f'\nAll results saved to Google Drive:')
print(f'  {DRIVE_BASE}/results/')
print('\nTo merge with local Mac results, download these CSVs and run:')
print('  python3 scripts/merge_results.py')